<a href="https://colab.research.google.com/github/srikanthkakumanu/do-hugging-face/blob/master/transformers_hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Transformers using GPT from Hugging Face Framework

## Loading the Model

There are two ways to load the models using **transformers library**.

Using AutoTokenizer + AutoModel &mdash; Gives fine-grained control over tokenization and model setup.   
Load the model via Pipeline

*   Using **AutoTokenizer + AutoModel**:
    *     Gives fined-grained control over **tokenization** and **model setup**.
    *     Full control over **tokenization, model execution** and **post-processing**. we must handle tokenization, inference, and decoding manually. Requires understanding of model input/output formats.
    *     Works for model **training**, **fine-tuning** and **inference**.

*   Using **Pipeline**:
    *     This high-level, quickstart approach, pipeline works as high-level helper.
    *     Automatically handles tokenization, model loading and output formatting.     
    *     Not ideal for model training and advanced workflows.
    *     Slightly overhead compared to manual loading.    

### Loading the Model Using Pipeline

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model='openai-community/gpt2') # task = text-generation

In [ ]:
prompt = "What is deep learning?"
output = pipe(prompt)

In [ ]:
print(output)
print(output[0]['generated_text'])

### Loading the Model using (AutoTokenizer + AutoModel)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2")

In [ ]:
print('Tokenizer: ', tokenizer)
print('Model Architecture: ', model)

### Tokenization

In [ ]:
sentence = 'unsure'
input_ids = tokenizer(sentence)['input_ids']
print(input_ids) # Return Token IDs
# input_ids = tokenizer(sentence, return_tensors='pt') # for PyTorch Tensor
# print(input_ids)
tokenizer.decode(input_ids[0]) # To get the text based on token id

sentence = 'unbelievable'
input_ids = tokenizer(sentence, return_tensors='pt').input_ids
print(input_ids) # Return Token IDs

In [ ]:
for token_id in input_ids[0]:
  print(tokenizer.decode(token_id))

un
bel
iev
able


In [ ]:
word = "homoscedasticity"
my_ids = tokenizer(word, return_tensors="pt").input_ids
my_ids

tensor([[26452,   418,   771,  3477,   414]])

In [ ]:
tokenizer.decode(my_ids.squeeze())

'homoscedasticity'

In [ ]:
word = "pneumonoultramicroscopicsilicovolcanoconiosis"
my_ids = tokenizer(word, return_tensors="pt").input_ids
# len(my_ids[0])
my_ids

tensor([[   79, 25668,   261, 25955,   859,  2500,  1416,   404,   873, 41896,
           709,   349,  5171, 36221, 42960]])

In [ ]:
for token_id in my_ids.squeeze():
    print(tokenizer.decode(token_id))

p
neum
on
oult
ram
icro
sc
op
ics
ilic
ov
ol
can
ocon
iosis


In [ ]:
sentence = "antidisestablishmentarianism"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
token_ids, len(token_ids[0])

(tensor([[  415, 29207, 44390,  3699,  1042]]), 5)

In [ ]:
word = "floccinaucinihilipilification"

my_ids = tokenizer(word, return_tensors="pt").input_ids
my_ids, len(my_ids[0])

(tensor([[ 2704, 13966,   259, 14272,   259, 20898,   541,   346,  2649]]), 9)

In [ ]:
for token_id in my_ids[0]:
    print(tokenizer.decode(token_id))

fl
occ
in
auc
in
ihil
ip
il
ification


### Text Generation and Prediction

In [ ]:
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
print('Model Architecture: ', gpt2)

### Predicting the Next Word

**Logits &mdash;** Logits are raw scores the model generates for each potential output in large language models (LLMs). While these logits are not probabilities, they offer valuable insight into how confident the model is about its predictions before they are transformed into final probabilities. A higher logit value indicates stronger confidence in a particular outcome, while lower values suggest more uncertainty.

*Below code block predicts the next word by GPT*.

In [ ]:
sentence = "I learn machine learning to" #  It guessed when we execute this block several times and it goes on: --be able to do things that--
# We have to tokenize the string before passing to any GPT LLM i.e. Data Processing Technique
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
# print(gpt2(token_ids).logits) # Its contain no. of vectors that the sentence have i.e. one word = one vector
outputs = gpt2(token_ids).logits[0, -1] # last word/token
tokenizer.decode(outputs.argmax()) # It guesses the next word

' be'

### Sampling Stratagies

**Temperature &mdash;** controls the randomness of token selection by scaling the logit values **before the softmax** function converts them into probabilities. **It is the single most important LLM sampling parameter**.

**Top-K &mdash;** Fixed number of candidate tokens. Sampling restricts the model's choice to the K most probable tokens, discarding everything else. It is simpler but less alternative to Top-P. **It is applied after softmax**.

**Top-P &mdash;** It is also called nucleus sampling. Cumulative probability threshold of tokens. **It is applied after softmax**. It dynamically selects the smallest set of tokens whose cumulative probability exceeds the threshold P.

In [ ]:
sentence = "I learn machine learning to enhance"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
outputs = gpt2(token_ids).logits[0, -1] # last word/token
final_logits = torch.topk(outputs, 20) # Feel free to play around with the K i.e. 20 is K

for index in final_logits.indices:
    print(tokenizer.decode(index))

 our
 the
 my
 your
 human
 a
 and
 its
 their
 performance
 learning
 user
 data
 social
 real
 understanding
 information
,
 intelligence
 people


In [ ]:
torch.softmax(final_logits.values, dim=0).sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [ ]:
def greedy_decode(logits):
    """Return token index with maximum probability."""
    return torch.argmax(logits, dim=-1) # -1 last dimention where maximum probability exists

# TOP K SAMPLING

def top_k_sampling(logits, k=50):
    """
    keeps only top-k logits, normalize them into probability.
    them sample one token from the filtered distribution.
    """
    values, indices = torch.topk(logits, k)
    probs = F.softmax(values, dim=-1)
    sampled = torch.multinomial(probs, 1)
    return indices[sampled]

# Top-p (Nuecles) Sampling

def top_p_sampling(logits, p=0.9):
    """
    Sort tokens by probability, keep smallest number whose culumative
    probability exceeds threshold p, then sample one token.
    """

    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = sorted_probs.cumsum(dim=-1)

    # Mask token outside nuclues
    mask = cumulative_probs > p
    sorted_logits[mask] = float("-inf")

    # Sample from filtered logits
    filtered_probs = F.softmax(sorted_logits, dim=-1)
    sampled = torch.multinomial(filtered_probs, 1)

    # Return token index in originial vocabulary
    return sorted_indices[sampled]

## Temperature Sampling ##

def temperature_sampling(logits, temperature=1.0):
    """
    Scale logits by temperature before sampling.
    Lower temperature => sharper distribution
    """

    scaled = logits / temperature
    probs = F.softmax(scaled, dim=-1)
    return torch.multinomial(probs, 1)


## Random Sampling ##

def random_sampling(logits):
    """
    Sample dirctly from softmax distribution without filtring
    """

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, 1)

# sentence = "Today I decided to go to the local library and find out what was in my wallet."
sentence = "I am really happy becuase I have gone back in time."
inputs = tokenizer(sentence, return_tensors="pt")
output = gpt2(**inputs)
logits = output.logits[0, -1]

print(f"Greedy Decode: ", tokenizer.decode([greedy_decode(logits)]))
print(f"Top-K Sampling: ", tokenizer.decode(top_k_sampling(logits, k=10)))
print(f"Top-P-Sampling: ", tokenizer.decode(top_p_sampling(logits, p=0.9)))
print(f"Temp: ", tokenizer.decode(temperature_sampling(logits, temperature=1)))
print(f"Random: ", tokenizer.decode(random_sampling(logits)))

Greedy Decode:   I
Top-K Sampling:   I
Top-P-Sampling:   Whatever
Temp:  

Random:   I


In [ ]:
tokenizer.decode(top_k_sampling(outputs))

' the'

In [ ]:
outputs

tensor([-109.4038, -108.0284, -114.4298,  ..., -111.9259, -117.0069,
        -109.9712], grad_fn=<SelectBackward0>)

In [ ]:
tokenizer.decode(top_p_sampling(outputs, p=0.9))

' user'

In [ ]:
tokenizer.decode(temperature_sampling(outputs, temperature=1.5))

' social'

In [ ]:
tokenizer.decode(random_sampling(outputs))

' my'

In [ ]:
sentence = "I learn machine learning to enhance our understanding of the brain in"
token_ids = tokenizer(sentence, return_tensors="pt").input_ids
outputs = gpt2(token_ids).logits # Raw Unnormlized Score - Values
outputs = torch.softmax(outputs[0, -1], dim=-1)

top10 = torch.topk(outputs, k=10)

for index, value in zip(top10.indices, top10.values):
    print(f"{tokenizer.decode(index)} -- {value:.1%}")

 a -- 14.7%
 the -- 7.7%
 order -- 6.2%
 ways -- 6.2%
 general -- 4.5%
 humans -- 2.9%
 many -- 2.8%
 an -- 2.4%
 our -- 2.3%
 everyday -- 1.7%


### Important References



*   https://amitray.com/llm-parameters-temperature-top-p-top-k-guide/#top-k
